# P5f-1 ワープ状態機械の試作
固定コミット未設定なら停止。各承認フラグは独立。関門1未合格ならGPUへ進まない。CUDAエラー後はランタイムを再起動する。採否はユーザーが決める。


In [ ]:
CODE_COMMIT = "f185ff8098345fd3763a8f2033fb8807ea31ac4c"
from pathlib import Path
import re, subprocess, sys, os
if not re.fullmatch(r"[0-9a-f]{40}", CODE_COMMIT):
    raise ValueError("固定コミット未設定。停止します")
from google.colab import drive
drive.mount('/content/drive')
REPO = Path('/content/viveMonte-p5g')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/gakugaku3333/chatcarlo.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', CODE_COMMIT], check=True)
if subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() != CODE_COMMIT:
    raise ValueError('コミット不一致')
os.chdir(REPO)
sys.path.insert(0, str(REPO))
from prototypes.mv_electron.gpu import p5g_colab, p5g_core
BUNDLE = Path('/content/drive/MyDrive/viveMonte/p5/p5b')
RESULTS = Path('/content/drive/MyDrive/viveMonte/p5/p5g')
EXPECTED_INPUT_SHA256 = '7b02611de118c11add0471744e64fc1b77cfb0aeb4fe285e4f29b9e7699d920a'
EXPECTED_REFERENCE_SHA256 = '02f01237752230937def6c9ffdecefbbab398d9d600a5ec890b7ad16a08b289f'
EXPECTED_SOURCE_SHA256 = '2ce5fe575ee6317b1be5cd3c0f319ed1cd9bc07b19fb104e1addcb11782832d0'
assert p5g_colab.d.BUNDLE_SHA == {'p5b_inputs.npz': EXPECTED_INPUT_SHA256, 'p5b_reference.npz': EXPECTED_REFERENCE_SHA256}
assert p5g_colab.d.SOURCE_SHA == EXPECTED_SOURCE_SHA256
p5g_colab.d.load_bundle(BUNDLE)
GATE1 = RESULTS/'gate1.json'  # Claudeの手元で実行した100バッチの記録をコピー
p5g_core.require_gate1(GATE1)


In [ ]:
APPROVE_MAP = False
if not APPROVE_MAP: raise ValueError('発散の地図の承認待ち')
p5g_colab.divergence_map(BUNDLE, RESULTS/'map_v0', GATE1)


In [ ]:
APPROVE_GATE2 = False
if not APPROVE_GATE2: raise ValueError('関門2の承認待ち')
p5g_colab.gate2(BUNDLE, RESULTS/'gate2', GATE1)


In [ ]:
APPROVE_GATE3 = False
if not APPROVE_GATE3: raise ValueError('関門3の承認待ち')
p5g_colab.gate3(BUNDLE, RESULTS/'gate3', GATE1, RESULTS/'gate2/gate2.json')


In [ ]:
APPROVE_TIMING = False
if not APPROVE_TIMING: raise ValueError('同一セッションの速さ比較の承認待ち')
p5g_colab.timing(BUNDLE, RESULTS/'timing', GATE1, RESULTS/'gate2/gate2.json', RESULTS/'gate3/gate3.json')


In [ ]:
APPROVE_COMPARISON_MAP = False
if not APPROVE_COMPARISON_MAP: raise ValueError('参加レーン比較の承認待ち')
p5g_colab.divergence_map(BUNDLE, RESULTS/'map_comparison', GATE1, compare=True, gate2_report=RESULTS/'gate2/gate2.json', gate3_report=RESULTS/'gate3/gate3.json')
